# Two-Stage DL + Ensemble ML Pipeline

**Architecture**: Deep Learning Backbones sebagai Embedding Extractors → Ensemble Machine Learning → Voting Classifier

**Stage 1**: Train DL backbones (EfficientNet-B2 untuk JENIS, EfficientNet-B0 untuk WARNA) dengan Global Average Pooling untuk extract embeddings

**Stage 2**: Train ensemble ML models (CatBoost, XGBoost, RandomForest, GradientBoosting, LogisticRegression) pada embeddings, lalu combine via Voting Classifier

**Benefits**:
- DL backbones: Extract powerful deep features dari images
- Ensemble ML: Better generalization, reduce overfitting, combine multiple perspectives
- Voting: Aggregate predictions dari multiple models untuk robust final decision

**Preprocessing**: Sama seperti pipeline sebelumnya (YOLO person detection → crop → augmentation)

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# Ensemble ML Models
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
from ultralytics import YOLO

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("\nEnsemble Models Available:")
print("  - CatBoost")
print("  - XGBoost")
print("  - RandomForest")
print("  - GradientBoosting")
print("  - LogisticRegression")

## 2. Load Data & Class Weights

In [ ]:
train_df = pd.read_csv('train.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Train data shape: {train_df.shape}")
print(f"Test data count: {len(sample_sub)}")
print("\nTrain data distribution:")
print(f"Jenis - Kaos: {(train_df['jenis']==0).sum()}, Hoodie: {(train_df['jenis']==1).sum()}")
print(f"Warna - Merah: {(train_df['warna']==0).sum()}, Kuning: {(train_df['warna']==1).sum()}, Biru: {(train_df['warna']==2).sum()}, Hitam: {(train_df['warna']==3).sum()}, Putih: {(train_df['warna']==4).sum()}")

# Class weights untuk JENIS (inverse frequency dengan smoothing)
total_jenis = len(train_df)
jenis_counts = train_df['jenis'].value_counts().sort_index().values
jenis_weights = total_jenis / (2 * jenis_counts)
jenis_weights = np.power(jenis_weights, 0.75)
jenis_weights_tensor = torch.FloatTensor(jenis_weights).to(device)
print(f"\nJenis weights (inverse freq + smoothing):")
print(f"  Kaos: {jenis_weights[0]:.4f}, Hoodie: {jenis_weights[1]:.4f}")

# Class weights untuk WARNA (class-balanced loss)
beta = 0.9999
warna_counts = train_df['warna'].value_counts().sort_index().values
effective_num = 1.0 - np.power(beta, warna_counts)
warna_weights = (1.0 - beta) / effective_num
warna_weights = warna_weights / warna_weights.sum() * len(warna_weights)
warna_weights_tensor = torch.FloatTensor(warna_weights).to(device)
print(f"\nWarna weights (class-balanced loss, beta={beta}):")
for i, color in enumerate(['Merah', 'Kuning', 'Biru', 'Hitam', 'Putih']):
    print(f"  {color}: {warna_weights[i]:.4f}")

## 3. YOLO Preprocessing (Sama seperti pipeline sebelumnya)

In [ ]:
def extract_clothing_region(image_path, yolo_model, conf_threshold=0.3):
    """
    Detect person region using YOLO dan crop clothing area
    Sama seperti preprocessing pipeline sebelumnya
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = yolo_model(img_rgb, verbose=False)
    
    best_box = None
    best_conf = 0
    
    for result in results:
        boxes = result.boxes
        for box in boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            
            if cls == 0 and conf > conf_threshold and conf > best_conf:
                best_conf = conf
                best_box = box.xyxy[0].cpu().numpy()
    
    if best_box is not None:
        x1, y1, x2, y2 = map(int, best_box)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img_rgb.shape[1], x2), min(img_rgb.shape[0], y2)
        cropped = img_rgb[y1:y2, x1:x2]
        return Image.fromarray(cropped)
    else:
        return Image.fromarray(img_rgb)

yolo_model = YOLO('yolov8n.pt')
print("YOLO model loaded (conf_threshold=0.3)")

## 4. Dataset Classes

In [ ]:
class JenisDataset(Dataset):
    """Dataset untuk klasifikasi JENIS (Kaos vs Hoodie)"""
    def __init__(self, df, img_dir, transform=None, yolo_model=None, use_yolo=True, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.yolo_model = yolo_model
        self.use_yolo = use_yolo
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['id']
        
        img_path = None
        for ext in ['.jpg', '.png']:
            path = os.path.join(self.img_dir, f'{img_id}{ext}')
            if os.path.exists(path):
                img_path = path
                break
        
        if img_path is None:
            raise FileNotFoundError(f"Image not found for id {img_id}")
        
        if self.use_yolo and self.yolo_model is not None:
            image = extract_clothing_region(img_path, self.yolo_model)
        else:
            image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        if self.is_test:
            return image, img_id
        else:
            jenis = self.df.iloc[idx]['jenis']
            return image, torch.tensor(jenis, dtype=torch.long)

class WarnaDataset(Dataset):
    """Dataset untuk klasifikasi WARNA (5 colors)"""
    def __init__(self, df, img_dir, transform=None, yolo_model=None, use_yolo=True, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.yolo_model = yolo_model
        self.use_yolo = use_yolo
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['id']
        
        img_path = None
        for ext in ['.jpg', '.png']:
            path = os.path.join(self.img_dir, f'{img_id}{ext}')
            if os.path.exists(path):
                img_path = path
                break
        
        if img_path is None:
            raise FileNotFoundError(f"Image not found for id {img_id}")
        
        if self.use_yolo and self.yolo_model is not None:
            image = extract_clothing_region(img_path, self.yolo_model)
        else:
            image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        if self.is_test:
            return image, img_id
        else:
            warna = self.df.iloc[idx]['warna']
            return image, torch.tensor(warna, dtype=torch.long)

# Transforms - SAMA SEPERTI PIPELINE SEBELUMNYA
jenis_train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

jenis_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

warna_train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

warna_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Dataset classes dan transforms ready")

## 5. Backbone Models untuk Embedding Extraction

In [ ]:
class JenisEmbeddingExtractor(nn.Module):
    """
    EfficientNet-B2 sebagai Embedding Extractor untuk JENIS
    
    Process:
    1. Backbone: EfficientNet-B2 features extraction
    2. Task-specific augmentation (training only): Dropout2d untuk spatial robustness
    3. Global Average Pooling: Reduce spatial dimensions → 1408-dim embedding
    4. Output: Embedding vector untuk Ensemble ML
    """
    def __init__(self, embedding_dim=1408, pretrained=True):
        super(JenisEmbeddingExtractor, self).__init__()
        
        backbone = models.efficientnet_b2(pretrained=pretrained)
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        self.embedding_dim = embedding_dim
        
        # Task-specific augmentation (training only)
        self.feature_dropout = nn.Dropout2d(0.1)
        
        # Auxiliary classifier untuk training embeddings
        self.aux_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )
    
    def forward(self, x, return_embedding=False):
        # Extract features
        features = self.features(x)
        
        # Task-specific augmentation (training only)
        if self.training:
            features = self.feature_dropout(features)
        
        # Global Average Pooling
        features = self.avgpool(features)
        embedding = torch.flatten(features, 1)
        
        if return_embedding:
            return embedding
        else:
            logits = self.aux_classifier(embedding)
            return logits, embedding

class WarnaEmbeddingExtractor(nn.Module):
    """
    EfficientNet-B0 sebagai Embedding Extractor untuk WARNA
    
    Process:
    1. Backbone: EfficientNet-B0 features extraction
    2. Task-specific augmentation (training only): Channel dropout + noise untuk color robustness
    3. Global Average Pooling: Reduce spatial dimensions → 1280-dim embedding
    4. Output: Embedding vector untuk Ensemble ML
    """
    def __init__(self, embedding_dim=1280, pretrained=True):
        super(WarnaEmbeddingExtractor, self).__init__()
        
        backbone = models.efficientnet_b0(pretrained=pretrained)
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        self.embedding_dim = embedding_dim
        
        # Task-specific augmentation (training only)
        self.channel_dropout = nn.Dropout2d(0.15)
        
        # Auxiliary classifier untuk training embeddings
        self.aux_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 5)
        )
    
    def forward(self, x, return_embedding=False):
        # Extract features
        features = self.features(x)
        
        # Task-specific augmentation (training only)
        if self.training:
            features = self.channel_dropout(features)
            noise = torch.randn_like(features) * 0.01
            features = features + noise
        
        # Global Average Pooling
        features = self.avgpool(features)
        embedding = torch.flatten(features, 1)
        
        if return_embedding:
            return embedding
        else:
            logits = self.aux_classifier(embedding)
            return logits, embedding

jenis_backbone = JenisEmbeddingExtractor(pretrained=True).to(device)
warna_backbone = WarnaEmbeddingExtractor(pretrained=True).to(device)

print("Backbone Models Initialized:")
print(f"  JENIS (EfficientNet-B2): {sum(p.numel() for p in jenis_backbone.parameters()):,} params → 1408-dim embeddings")
print(f"  WARNA (EfficientNet-B0): {sum(p.numel() for p in warna_backbone.parameters()):,} params → 1280-dim embeddings")

## 6. Data Loaders

In [ ]:
# Split data
train_data, val_data = train_test_split(
    train_df, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_df['jenis']
)

# JENIS Loaders
jenis_train_dataset = JenisDataset(train_data, 'train/train', jenis_train_transform, yolo_model, use_yolo=True)
jenis_val_dataset = JenisDataset(val_data, 'train/train', jenis_test_transform, yolo_model, use_yolo=True)

jenis_train_loader = DataLoader(jenis_train_dataset, batch_size=32, shuffle=True, num_workers=0)
jenis_val_loader = DataLoader(jenis_val_dataset, batch_size=32, shuffle=False, num_workers=0)

# WARNA Loaders
warna_train_dataset = WarnaDataset(train_data, 'train/train', warna_train_transform, yolo_model, use_yolo=True)
warna_val_dataset = WarnaDataset(val_data, 'train/train', warna_test_transform, yolo_model, use_yolo=True)

warna_train_loader = DataLoader(warna_train_dataset, batch_size=32, shuffle=True, num_workers=0)
warna_val_loader = DataLoader(warna_val_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Data Loaders Ready:")
print(f"  JENIS - Train: {len(jenis_train_dataset)}, Val: {len(jenis_val_dataset)}")
print(f"  WARNA - Train: {len(warna_train_dataset)}, Val: {len(warna_val_dataset)}")

## STAGE 1: Train Backbones untuk Learn Discriminative Embeddings

Train deep learning backbones dengan auxiliary classifier untuk learn embeddings yang discriminative. Embeddings ini akan digunakan sebagai input untuk ensemble ML models.

In [ ]:
# Training Setup
jenis_criterion = nn.CrossEntropyLoss(weight=jenis_weights_tensor)
jenis_optimizer = optim.AdamW(jenis_backbone.parameters(), lr=0.001, weight_decay=0.01)
jenis_scheduler = optim.lr_scheduler.ReduceLROnPlateau(jenis_optimizer, mode='min', factor=0.5, patience=3, verbose=True)

warna_criterion = nn.CrossEntropyLoss(weight=warna_weights_tensor)
warna_optimizer = optim.AdamW(warna_backbone.parameters(), lr=0.001, weight_decay=0.01)
warna_scheduler = optim.lr_scheduler.ReduceLROnPlateau(warna_optimizer, mode='min', factor=0.5, patience=3, verbose=True)

def train_backbone_epoch(model, loader, criterion, optimizer, device):
    """Train backbone untuk 1 epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits, embeddings = model(images, return_embedding=False)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def validate_backbone(model, loader, criterion, device):
    """Validate backbone"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            logits, embeddings = model(images, return_embedding=False)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print("Training functions ready")

### 7.1 Train JENIS Backbone

In [ ]:
print("Training JENIS Backbone (EfficientNet-B2)...")
print("=" * 60)

num_epochs = 10
best_val_loss = float('inf')

for epoch in range(num_epochs):
    train_loss, train_acc = train_backbone_epoch(jenis_backbone, jenis_train_loader, jenis_criterion, jenis_optimizer, device)
    val_loss, val_acc = validate_backbone(jenis_backbone, jenis_val_loader, jenis_criterion, device)
    
    jenis_scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
    print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(jenis_backbone.state_dict(), 'jenis_backbone.pth')
        print(f"  Model saved (best val loss: {best_val_loss:.4f})")
    print()

print("JENIS Backbone Training Complete")
print(f"Best validation loss: {best_val_loss:.4f}")

### 7.2 Train WARNA Backbone

In [ ]:
print("Training WARNA Backbone (EfficientNet-B0)...")
print("=" * 60)

num_epochs = 10
best_val_loss = float('inf')

for epoch in range(num_epochs):
    train_loss, train_acc = train_backbone_epoch(warna_backbone, warna_train_loader, warna_criterion, warna_optimizer, device)
    val_loss, val_acc = validate_backbone(warna_backbone, warna_val_loader, warna_criterion, device)
    
    warna_scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
    print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(warna_backbone.state_dict(), 'warna_backbone.pth')
        print(f"  Model saved (best val loss: {best_val_loss:.4f})")
    print()

print("WARNA Backbone Training Complete")
print(f"Best validation loss: {best_val_loss:.4f}")

## STAGE 2: Extract Embeddings dari Trained Backbones

Gunakan trained backbones untuk extract embeddings dari semua data (train, validation, test). Embeddings ini akan menjadi input untuk ensemble ML models.

In [ ]:
# Load best models
jenis_backbone.load_state_dict(torch.load('jenis_backbone.pth'))
warna_backbone.load_state_dict(torch.load('warna_backbone.pth'))
jenis_backbone.eval()
warna_backbone.eval()

def extract_embeddings(model, loader, device):
    """Extract embeddings dari model"""
    embeddings_list = []
    labels_list = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            embeddings = model(images, return_embedding=True)
            embeddings_list.append(embeddings.cpu().numpy())
            labels_list.append(labels.numpy())
    
    embeddings = np.vstack(embeddings_list)
    labels = np.concatenate(labels_list)
    return embeddings, labels

print("Extracting embeddings...")
print("=" * 60)

# Extract JENIS embeddings
print("Extracting JENIS embeddings...")
X_train_jenis, y_train_jenis = extract_embeddings(jenis_backbone, jenis_train_loader, device)
X_val_jenis, y_val_jenis = extract_embeddings(jenis_backbone, jenis_val_loader, device)
print(f"  Train: {X_train_jenis.shape}, Val: {X_val_jenis.shape}")

# Extract WARNA embeddings
print("Extracting WARNA embeddings...")
X_train_warna, y_train_warna = extract_embeddings(warna_backbone, warna_train_loader, device)
X_val_warna, y_val_warna = extract_embeddings(warna_backbone, warna_val_loader, device)
print(f"  Train: {X_train_warna.shape}, Val: {X_val_warna.shape}")

print("\nEmbeddings extraction complete!")
print(f"JENIS: {X_train_jenis.shape[1]}-dimensional embeddings")
print(f"WARNA: {X_train_warna.shape[1]}-dimensional embeddings")

## STAGE 3: Train Ensemble ML Models

Train multiple ML models (CatBoost, XGBoost, RandomForest, GradientBoosting, LogisticRegression) pada embeddings untuk each task. Ensemble ini akan di-combine menggunakan Voting Classifier.

### 8.1 Train JENIS Ensemble Models

In [ ]:
print("Training JENIS Ensemble Models...")
print("=" * 60)

# Initialize models
jenis_catboost = CatBoostClassifier(iterations=1000, depth=6, learning_rate=0.03, verbose=0, random_state=42)
jenis_xgboost = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, random_state=42, eval_metric='logloss')
jenis_rf = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1)
jenis_gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=5, random_state=42)
jenis_lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)

jenis_models = {
    'CatBoost': jenis_catboost,
    'XGBoost': jenis_xgboost,
    'RandomForest': jenis_rf,
    'GradientBoosting': jenis_gb,
    'LogisticRegression': jenis_lr
}

# Train and evaluate each model
for name, model in jenis_models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_jenis, y_train_jenis)
    
    train_acc = accuracy_score(y_train_jenis, model.predict(X_train_jenis))
    val_acc = accuracy_score(y_val_jenis, model.predict(X_val_jenis))
    
    print(f"  Train Accuracy: {train_acc*100:.2f}%")
    print(f"  Val Accuracy:   {val_acc*100:.2f}%")
    
    joblib.dump(model, f'jenis_{name.lower()}.pkl')
    print(f"  Model saved: jenis_{name.lower()}.pkl")

print("\nJENIS Ensemble Models Training Complete!")

### 8.2 Train WARNA Ensemble Models

In [ ]:
print("Training WARNA Ensemble Models...")
print("=" * 60)

# Initialize models
warna_catboost = CatBoostClassifier(iterations=1000, depth=6, learning_rate=0.03, verbose=0, random_state=42)
warna_xgboost = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, random_state=42, eval_metric='mlogloss')
warna_rf = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1)
warna_gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=5, random_state=42)
warna_lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42, multi_class='multinomial')

warna_models = {
    'CatBoost': warna_catboost,
    'XGBoost': warna_xgboost,
    'RandomForest': warna_rf,
    'GradientBoosting': warna_gb,
    'LogisticRegression': warna_lr
}

# Train and evaluate each model
for name, model in warna_models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_warna, y_train_warna)
    
    train_acc = accuracy_score(y_train_warna, model.predict(X_train_warna))
    val_acc = accuracy_score(y_val_warna, model.predict(X_val_warna))
    
    print(f"  Train Accuracy: {train_acc*100:.2f}%")
    print(f"  Val Accuracy:   {val_acc*100:.2f}%")
    
    joblib.dump(model, f'warna_{name.lower()}.pkl')
    print(f"  Model saved: warna_{name.lower()}.pkl")

print("\nWARNA Ensemble Models Training Complete!")

## STAGE 4: Create Voting Classifiers & Evaluate

Combine 5 models per task menggunakan Soft Voting (average probabilities) untuk final robust predictions.

In [ ]:
print("Creating Voting Classifiers...")
print("=" * 60)

# JENIS Voting Classifier
jenis_voting = VotingClassifier(
    estimators=[
        ('catboost', jenis_catboost),
        ('xgboost', jenis_xgboost),
        ('rf', jenis_rf),
        ('gb', jenis_gb),
        ('lr', jenis_lr)
    ],
    voting='soft'
)
print("\nTraining JENIS Voting Classifier...")
jenis_voting.fit(X_train_jenis, y_train_jenis)
jenis_val_pred = jenis_voting.predict(X_val_jenis)
jenis_val_acc = accuracy_score(y_val_jenis, jenis_val_pred)
print(f"JENIS Voting Classifier Val Accuracy: {jenis_val_acc*100:.2f}%")
joblib.dump(jenis_voting, 'jenis_voting.pkl')

# WARNA Voting Classifier
warna_voting = VotingClassifier(
    estimators=[
        ('catboost', warna_catboost),
        ('xgboost', warna_xgboost),
        ('rf', warna_rf),
        ('gb', warna_gb),
        ('lr', warna_lr)
    ],
    voting='soft'
)
print("\nTraining WARNA Voting Classifier...")
warna_voting.fit(X_train_warna, y_train_warna)
warna_val_pred = warna_voting.predict(X_val_warna)
warna_val_acc = accuracy_score(y_val_warna, warna_val_pred)
print(f"WARNA Voting Classifier Val Accuracy: {warna_val_acc*100:.2f}%")
joblib.dump(warna_voting, 'warna_voting.pkl')

# Calculate Exact Match Ratio
exact_match = np.sum((jenis_val_pred == y_val_jenis) & (warna_val_pred == y_val_warna))
exact_match_ratio = exact_match / len(y_val_jenis)
print("\n" + "=" * 60)
print(f"VALIDATION RESULTS:")
print(f"  JENIS Accuracy:      {jenis_val_acc*100:.2f}%")
print(f"  WARNA Accuracy:      {warna_val_acc*100:.2f}%")
print(f"  EXACT MATCH RATIO:   {exact_match_ratio*100:.2f}%")
print("=" * 60)

print("\nVoting Classifiers saved!")
print("  - jenis_voting.pkl")
print("  - warna_voting.pkl")

## 9. Generate Test Predictions

Extract embeddings dari test data, lalu predict menggunakan voting classifiers untuk generate final submission.

In [ ]:
# Create test datasets
jenis_test_dataset = JenisDataset(sample_sub, 'test/test', jenis_test_transform, yolo_model, use_yolo=True, is_test=True)
warna_test_dataset = WarnaDataset(sample_sub, 'test/test', warna_test_transform, yolo_model, use_yolo=True, is_test=True)

jenis_test_loader = DataLoader(jenis_test_dataset, batch_size=32, shuffle=False, num_workers=0)
warna_test_loader = DataLoader(warna_test_dataset, batch_size=32, shuffle=False, num_workers=0)

print("Extracting test embeddings...")
print("=" * 60)

# Extract test embeddings
def extract_test_embeddings(model, loader, device):
    """Extract embeddings dan IDs dari test data"""
    embeddings_list = []
    ids_list = []
    
    with torch.no_grad():
        for images, img_ids in loader:
            images = images.to(device)
            embeddings = model(images, return_embedding=True)
            embeddings_list.append(embeddings.cpu().numpy())
            ids_list.extend([int(img_id) for img_id in img_ids])
    
    embeddings = np.vstack(embeddings_list)
    return embeddings, ids_list

# Extract JENIS test embeddings
X_test_jenis, jenis_ids = extract_test_embeddings(jenis_backbone, jenis_test_loader, device)
print(f"JENIS test embeddings: {X_test_jenis.shape}")

# Extract WARNA test embeddings
X_test_warna, warna_ids = extract_test_embeddings(warna_backbone, warna_test_loader, device)
print(f"WARNA test embeddings: {X_test_warna.shape}")

# Verify IDs match
assert jenis_ids == warna_ids, "Test IDs mismatch!"
test_ids = jenis_ids

# Predict using voting classifiers
print("\nGenerating predictions using Voting Classifiers...")
jenis_pred = jenis_voting.predict(X_test_jenis)
warna_pred = warna_voting.predict(X_test_warna)

# Create submission
submission = pd.DataFrame({
    'id': test_ids,
    'jenis': jenis_pred,
    'warna': warna_pred
})

submission.to_csv('submission_ensemble.csv', index=False)
print("\nSubmission saved: submission_ensemble.csv")
print(f"Total predictions: {len(submission)}")
print("\nSample predictions:")
print(submission.head(10))

## Pipeline Summary

**Complete Two-Stage DL + Ensemble ML Pipeline:**

**STAGE 1: Train DL Backbones**
- JENIS: EfficientNet-B2 trained untuk extract 1408-dim embeddings
- WARNA: EfficientNet-B0 trained untuk extract 1280-dim embeddings
- Backbones trained dengan auxiliary classifier dan class weights
- Preprocessing: YOLO person detection + crop + augmentation

**STAGE 2: Extract Embeddings**
- Gunakan Global Average Pooling untuk convert feature maps ke vectors
- Extract embeddings dari train, validation, dan test data
- Embeddings = representasi numerik high-level dari images

**STAGE 3: Train Ensemble ML**
- 5 models per task: CatBoost, XGBoost, RandomForest, GradientBoosting, LogisticRegression
- Each model trained pada embeddings (bukan raw images)
- Total 10 models (5 untuk JENIS, 5 untuk WARNA)

**STAGE 4: Voting Classifier**
- Soft voting: Average probabilities dari 5 models
- Final prediction = consensus dari multiple perspectives
- More robust dan generalize better daripada single model

**Expected Benefits:**
- DL backbones: Powerful deep feature extraction
- Ensemble ML: Reduce overfitting, combine strengths
- Voting: Robust decision making via consensus
- Target: Exact Match Ratio > 90%

**Output Files:**
- jenis_backbone.pth, warna_backbone.pth (DL models)
- jenis_*.pkl, warna_*.pkl (5 ensemble models per task)
- jenis_voting.pkl, warna_voting.pkl (voting classifiers)
- submission_ensemble.csv (final predictions)